# IC 4040 - Image Reducer

<div class="alert alert-block alert-info">
    <b>Note:</b> This notebook should be run with the <span style="font-family: 'Ariel', monospace;">stenv</span> environment.
</div>

The purpose of this notebook is to reduce the FLC files from Hubble by:

1. Aligning FLCs to the GAIA catalog
2. Drizzling Images together from a particular filter

## Imports

In [ ]:
# Python Imports
import logging
import os
import warnings
from pathlib import Path

# Astropy Colab Imports
from astropy import units as u
from astropy.io import fits
from astropy.table import QTable
from astropy.coordinates import SkyCoord
from astropy.stats import sigma_clipped_stats
# from photutils.detection import IRAFStarFinder
# from drizzlepac import updatehdr
from drizzlepac.tweakreg import TweakReg
from drizzlepac.astrodrizzle import AstroDrizzle
# from stwcs.wcsutil import HSTWCS

# 3rd Party Imports
import numpy as np
from tqdm.notebook import tqdm


## Notebook Setup

In [ ]:
# Check Directory
if Path.cwd().name != "Images":
    if Path.cwd().name == "Notebooks":
        os.chdir("../Images")
    else:
        raise RuntimeError("This notebook must be run from the Images directory.")

# Configure Logging
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.info("Current Directory: %s", Path.cwd())

In [ ]:
# Data Directory
DATA_DIR = Path('RawImages/wfc3')

# FLC Glob Pattern
FLC_CR_GLOB_PAT = '*crclean_flc.fits'

# Define Max Sep
MAX_SEP = 0.8 * u.arcsec

## Functions

In [ ]:
def get_xy_offset(
    fn: Path,
    image_matches: SkyCoord,
    gaia_matches: SkyCoord,
) -> tuple[u.Quantity, u.Quantity]:
    """
    Calculate the pixel shift to pass to updatewcs_with_shift to align the
    image to GAIA.

    updatewcs_with_shift subtracts [xsh, ysh] from the reference-frame pixel
    position of the chip CRPIX before converting back to a new CRVAL.  So
    xsh/ysh must be the correction to apply — i.e. how far to move the image
    sources toward GAIA (GAIA − image).  This function returns
    xsh = x_shifted − x_ref  (where "shifted" means the CRPIX displaced by the
    mean GAIA offset).

    Parameters:
    -----------
    fn : Path
        Filename of the image (used for logging).
    image_matches : SkyCoord
        Per-extension SkyCoord arrays for matched image sources.
    gaia_matches : SkyCoord
        Per-extension SkyCoord arrays for matched GAIA sources.

    Returns:
    --------
    xsh : float
        X shift in pixels for updatewcs_with_shift (extension 1 frame).
    ysh : float
        Y shift in pixels for updatewcs_with_shift (extension 1 frame).
    """

    if len(image_matches) == 0:
        logger.warning("No GAIA matches found for %s.", fn.name)
        return 0.0, 0.0

    # Sigma-clipped mean offset: image → GAIA (this is the correction direction)
    # A 3-sigma clip removes outliers from mismatches before averaging.
    dra, ddec = image_matches.spherical_offsets_to(gaia_matches)
    _, med_dra, _ = sigma_clipped_stats(dra.to_value(u.arcsec), sigma=1.0)    # Output: mean, median, stddev
    if np.isnan(med_dra):
        _, med_dra, _ = sigma_clipped_stats(dra.to_value(u.arcsec), sigma=2.0)    # Output: mean, median, stddev
    _, med_ddec, _ = sigma_clipped_stats(ddec.to_value(u.arcsec), sigma=1.0)  # Output: mean, median, stddev
    if np.isnan(med_ddec):
        _, med_ddec, _ = sigma_clipped_stats(ddec.to_value(u.arcsec), sigma=2.0)  # Output: mean, median, stddev
    med_dra <<= u.arcsec
    med_ddec <<= u.arcsec
    logger.info(
        "Sigma-clipped median offset for %s: dRA=%.4f\", dDec=%.4f\"",

        fn.name, med_dra.value, med_ddec.value,

    )
    return med_dra, med_ddec

## Load the Data

In [ ]:
# Get the File Names and Sort them by filter
fileNameDict = {}
for fn in DATA_DIR.rglob(FLC_CR_GLOB_PAT):

    # Open the file to get the filter
    with fits.open(fn) as hduList:
        hdr = hduList[0].header  # Get the Header
        if 'FILTER' in hdr:      # If the FILTER keyword exists (WFC3)
            filt = hdr['FILTER']
        elif 'CLEAR' not in hdr['FILTER1']:  # If FILTER1 is not clear (ACS)
            filt = hdr['FILTER1']
        else:                                # Else FILTER2 must be the filter (ACS)
            filt = hdr['FILTER2']

    # Store the Name using the filter as the dict key
    # Start the Empty List if Key does not exist
    if filt not in fileNameDict:
        fileNameDict[filt] = []
    fileNameDict[filt].append(fn)
    logger.debug("Added file %s to filter %s list.", fn.name, filt)
# fileNameDict

## Align Images to GAIA

Sometimes, the FLCs have so many CRs and so few Milky Way stars, it makes aligning the images difficult.
In this case, a different strategy is to align the Hubble Pipeline DRCs/DRZs to GAIA using `TweakReg` then
assigning the found WCS solution to the associated FLCs/FLTs using `TweakBack`.

For an example of this strategy, consider the work documented in the [Abell 1367](https://github.com/wwaldron/a1367)
repo on GitHub where the [Image Reducer Notebook](https://github.com/wwaldron/a1367/blob/main/Images/ImageReducer.ipynb)
implements this methodology.

### TweakReg Data Quality Flags

This cell defines the [ACS DQ flags](https://www.stsci.edu/hst/instrumentation/acs/data-analysis/dq-flag-definitions) we
want to ignore in the TweakReg process. The DQ flags that are most often used are:

* [ACS DQ Flags](https://www.stsci.edu/hst/instrumentation/acs/data-analysis/dq-flag-definitions)
* [WFC3-UVIS DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-2-uvis-data-calibration-steps#id-3.2UVISDataCalibrationSteps-3.2.3DataQualityArrayInitialization)
* [WFC3-IR DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-3-ir-data-calibration-steps#id-3.3IRDataCalibrationSteps-3.3.1DataQualityInitialization)

For an example of how to implement multiple DQ flags, consider the
[Image Reducer for ESO 137-001](https://github.com/wwaldron/ESO-137-001/blob/main/Images/ImageReducer.ipynb).

<div class="alert alert-block alert-info">
    <b>Note:</b> In the cells below where <span style="font-family: 'Ariel', monospace;">TweakReg</span> is called,
    the user <i>must</i> update the <span style="font-family: 'Ariel', monospace;">updatehdr</span>
    keyword to <span style="font-family: 'Ariel', monospace;">True</span> and rerun the cell once a valid
    WCS solution is found.
    If the value is left as <span style="font-family: 'Ariel', monospace;">False</span>, the header in the
    input file will not be updated.
</div>

In [ ]:
# DQ Bits
DQ_FILLED   = 2
DQ_BAD_DET  = 4
DQ_HOT_PIX  = 16
DQ_CR_PIX   = 4096+8192#+16384
DQ_GOOD_PIX = ~(DQ_FILLED + DQ_BAD_DET + DQ_HOT_PIX + DQ_CR_PIX) # Ignore these

In [ ]:
# # Load GAIA Coordinates
# GAIA_CRD_FILE = Path('../Data/GAIA/IC4040-GAIA-AlignmentStars-Coordinates.ecsv')
# gaia_crds = QTable.read(GAIA_CRD_FILE, format='ascii.ecsv')['SkyCoord']

In [ ]:
# # Setup the Star Finder
# star_finder = IRAFStarFinder(
#     threshold=5.0,   # Minimum signal-to-noise ratio for detection
#     fwhm=3.5,        # Full Width at Half Maximum of the stars
#     sharplo=0.35,    # Lower limit for the sharpness of the stars
#     sharphi=3,       # Upper limit for the sharpness of the stars
#     roundlo=0,       # Lower limit for the roundness of the stars
#     roundhi=0.4      # Upper limit for the roundness of the stars
# )

# # Setup Output QTable
# shifts = QTable(names=['filename', 'x_shift', 'y_shift'], dtype=[str, float, float])
# shifts['x_shift'].unit = u.deg
# shifts['y_shift'].unit = u.deg

# # Loop Through Files
# # Find Sources
# # Match to GAIA
# # Manually Align to GAIA
# for fn in (
#     file_name
#     for file_names in tqdm(fileNameDict.values(), desc="Processing Filters", unit="Filters")
#     for file_name in tqdm(file_names, desc="Processing Files", unit="Files", leave=False)
# ):
#     with fits.open(fn) as hdu_list:

#         # Loop through Extension Sets
#         image_matches, gaia_matches = [], []
#         for ext in [1, 2]:

#             # Get the Data and Header
#             data = hdu_list['SCI', ext].data
#             sci_hstwcs = HSTWCS(hdu_list, ext=('SCI', ext))

#             # Get the DQ Array and Mask it
#             dq = hdu_list['DQ', ext].data
#             mask = (dq & DQ_GOOD_PIX) != 0

#             # Get Star Locations
#             # Suppress photutils RuntimeWarning for NaN moments in faint/edge sources
#             with warnings.catch_warnings():
#                 warnings.filterwarnings(
#                     'ignore',
#                     category=RuntimeWarning,
#                     message='invalid value encountered in divide',
#                 )
#                 image_sources = star_finder(data, mask=mask)

#             # Filter Sources and Convert to SkyCoord
#             image_sources = image_sources[image_sources['npix'] < 5000]
#             image_crds = SkyCoord.from_pixel(
#                 image_sources['xcentroid'],
#                 image_sources['ycentroid'],
#                 sci_hstwcs,
#             )
#             logger.info("Found %d sources in %s extension %d.", len(image_crds), fn.name, ext)

#             # Match to GAIA
#             idx, sep2d, _ = gaia_crds.match_to_catalog_sky(image_crds)
#             sep_mask = sep2d <= MAX_SEP
#             gaia_matches.append(gaia_crds[sep_mask])
#             image_matches.append(image_crds[idx[sep_mask]])
#             logger.info("    Matched %d sources to GAIA extension %d.", sum(sep_mask), ext)

#     # Calculate the Mean Offset and Update the Header
#     xsh, ysh = get_xy_offset(fn, SkyCoord(image_matches), SkyCoord(gaia_matches))

#     # Apply the Shift to the Header
#     updatehdr.updatewcs_with_shift(
#         str(fn), str(fn), xsh=xsh.to_value('deg'), ysh=ysh.to_value('deg'), rot=0, scale=1,
#         wcsname='GAIA', reusename=True, force=True
#     )

#     # Add to Output Table
#     shifts.add_row([fn.name, xsh, ysh])

# # Write Table to File
# OUTPUT_TABLE_FILE = Path('Shifts/ManualAlignmentShifts.ecsv')
# shifts.write(OUTPUT_TABLE_FILE, format='ascii.ecsv', overwrite=True)

In [ ]:
# Align Best F814W Image to GAIA
# Setup Image Find Params
conv_width = 3.5
imagefindcfg = dict(
    # peakmax=900,
    threshold=9.0,
    conv_width=conv_width,
    dqbits=DQ_GOOD_PIX
)

# Run TweakReg on the F814W Images
MAIN_IMAGE = Path('RawImages/wfc3/visit06_parallel/ifhs06gvq_crclean_flc.fits')
Path(f"TweakReg-GAIA-Main-F814W.log").write_text("", encoding="utf-8")
TweakReg(
    str(MAIN_IMAGE),
    updatehdr=True,
    wcsname='GAIA',
    clean=True,
    configobj=None,
    shiftfile=True,
    outshifts=f'GAIA-Main-F814W.txt',
    outwcs=f'GAIA-Main-F814W.fits',
    refcat='../Data/GAIA/IC4040-GAIA-RefCatalog-icrs.txt',
    runfile=f'TweakReg-GAIA-Main-F814W.log',
    # use2dhist=False,
    see2dplot=False,
    searchrad=0.35,
    # xoffset=-1.5,
    # yoffset=-1.5,
    # fitgeometry='shift',
    residplot='No Plot',
    minobj=3,
    tolerance=3,
    imagefindcfg=imagefindcfg,
    interactive=False
)

In [ ]:
# Loop through Filters
refimagefindcfg = imagefindcfg.copy()
for filter, fileList in fileNameDict.items():

    # Image Dep Search Parameters
    threshold  = {
        'F814W': 9.0,
        'F475W': 7.5
    }[filter]

    # Image Find Parameters
    imagefindcfg = dict(
        # peakmax=900,
        threshold=threshold,
        conv_width=conv_width,
        dqbits=DQ_GOOD_PIX
    )

    # Align the Images to the GAIA data
    Path(f"TweakReg-GAIA-{filter}.log").write_text("", encoding="utf-8")
    TweakReg(
        [str(fn) for fn in fileList if MAIN_IMAGE.name != fn.name],
        updatehdr=True,
        wcsname='GAIA',
        clean=True,
        configobj=None,
        shiftfile=True,
        outshifts=f'GAIA-{filter}.txt',
        outwcs=f'GAIA-{filter}.fits',
        refimage=str(MAIN_IMAGE),
        runfile=f'TweakReg-GAIA-{filter}.log',
        # use2dhist=False,
        see2dplot=False,
        searchrad=0.2,
        # xoffset=0.0,
        # yoffset=0.0,
        # fitgeometry='shift',
        residplot='No Plot',
        minobj=15,
        tolerance=1,
        imagefindcfg=imagefindcfg,
        refimagefindcfg=refimagefindcfg,
        interactive=False
    )

In [ ]:
%%bash
# Move Log Files
mkdir -p logs/tweakreg Shifts
mv *.log logs/tweakreg

# Remove Intermediate Files
rm *.coo

# Move Shift Files
mv GAIA*.txt Shifts
mv GAIA*.fits Shifts

## Drizzle Images for CR Correction

Although there will be additional notes added later, it is worth noting that according to
[STScI](https://hst-docs.stsci.edu/drizzpac/chapter-6-reprocessing-with-the-drizzlepac-package/6-3-running-astrodrizzle#id-6.3RunningAstroDrizzle-SelectingtheOptimalScaleandPixfrac):

1. For sub-pixel dithered data, select an output scale that's smaller than the native scale.
It will even help in the cosmic ray rejection step.
1. A smaller final_pixfrac gives higher resolution and lower correlated noise, but also reduces
sensitivity to low-surface brightness features (though it is possible to convolve a high resolution
image later to go after low surface brightness features).
1. Keep the standard deviation of the weight map over the main part of the image to above ~0.3 of
the mean to insure that one does not lose significant signal-to-noise in ignoring the weight map in
final photometry.

To summarize the last step, a `final_scale`/`final_pixfrac` combo should be chosen such that,
for the weight image,
\begin{equation}
    \mathrm{std} \gtrsim 0.3 \, \mathrm{mean}
\end{equation}

### AstroDrizzle ACS Data Quality Flags

* [ACS DQ Flags](https://www.stsci.edu/hst/instrumentation/acs/data-analysis/dq-flag-definitions)
* [WFC3-UVIS DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-2-uvis-data-calibration-steps#id-3.2UVISDataCalibrationSteps-3.2.3DataQualityArrayInitialization)
* [WFC3-IR DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-3-ir-data-calibration-steps#id-3.3IRDataCalibrationSteps-3.3.1DataQualityInitialization)

In [ ]:
# DQ Bits
DQ_WARM_PIX = 64
DQ_BAD_COL  = 128
DQ_FULL_WELL= 256
DQ_SINK_PIX = 1024
DQ_GOOD_PIX = DQ_WARM_PIX + DQ_BAD_COL + DQ_FULL_WELL + DQ_SINK_PIX # Make these OK

### Drizzle F814W Images

In [ ]:
# Drizzle Images
AstroDrizzle(
    [str(fn) for fn in fileNameDict['F814W']],
    output='IC4040-F814W',
    runfile='F814W-Astro.log',
    wcskey='GAIA',
    context=False,
    configobj=None,
    num_cores=8,
    in_memory=False,
    build=True,
    restore=False,
    preserve=False,
    clean=True,
    skymethod='globalmin+match',
    driz_sep_scale=0.03,
    driz_sep_bits=DQ_GOOD_PIX,
    combine_type='median',
    combine_nhigh=1,
    combine_nlow=1,
    driz_cr_corr=False,
    final_wht_type='IVM',
    final_pixfrac=0.4,
    final_bits=DQ_GOOD_PIX,
    final_wcs=True,
    final_rot=0,
    final_scale=0.03
)

### Drizzle F475W Images

In [ ]:
# Drizzle Images
AstroDrizzle(
    [str(fn) for fn in fileNameDict['F475W']],
    output='IC4040-F475W',
    runfile='F475W-Astro.log',
    wcskey='GAIA',
    context=False,
    configobj=None,
    num_cores=8,
    in_memory=False,
    build=True,
    restore=False,
    preserve=False,
    clean=True,
    skymethod='globalmin+match',
    driz_sep_scale=0.03,
    driz_sep_bits=DQ_GOOD_PIX,
    combine_type='median',
    combine_nhigh=1,
    combine_nlow=1,
    driz_cr_corr=False,
    final_wht_type='IVM',
    final_pixfrac=0.4,
    final_bits=DQ_GOOD_PIX,
    final_wcs=True,
    final_refimage='IC4040-F814W_drc.fits'
)

In [ ]:
%%bash
# Move Log Files
mkdir -p logs/astrodrizzle
mv *.log logs/astrodrizzle

# Move Final Drizzled Images
mkdir -p ProcessedImages/HST/Drizzled
mv *_dr?.fits ProcessedImages/HST/Drizzled/